PRUEBA DE CONSISTENCIA DEL DATASET CON LA DOCUMENTACIÓN

In [1]:
# ============================================
# VALIDACIÓN DEL DATASET LAPOP BOLIVIA 2023
# ============================================
import pandas as pd
import numpy as np

print("=" * 70)
print("VALIDACIÓN DEL DATASET LAPOP BOLIVIA 2023")
print("=" * 70)

# --- CARGA ---
file_path = "../../data/raw/lapop_bolivia_2023.dta"
df = pd.read_stata(file_path, convert_categoricals=False)
print(f"\n✓ Dataset cargado: {df.shape[0]} registros × {df.shape[1]} variables")

# ============================================
# 1. VALIDACIÓN DE DISTRIBUCIÓN MUESTRAL
# ============================================
print("\n" + "=" * 70)
print("1. VALIDACIÓN DE DISTRIBUCIÓN MUESTRAL")
print("=" * 70)

# Distribución por estratos (comparar con Tabla 1 del reporte técnico)
print("\nDistribución por regiones (comparar con reporte técnico):")
distribucion_esperada = {
    1001: ("La Paz", 371),
    1002: ("Santa Cruz", 365),
    1003: ("Cochabamba", 294),
    1010: ("Beni-Pando", 192),
    1011: ("Potosí-Oruro", 241),
    1012: ("Chuquisaca-Tarija", 243)
}

print(f"\n{'Estrato':<6} {'Región':<20} {'Esperado':>10} {'Obtenido':>10} {'Diferencia':>12}")
print("-" * 70)

for estrato_code, (nombre, esperado) in distribucion_esperada.items():
    obtenido = (df['estratopri'] == estrato_code).sum()
    diferencia = obtenido - esperado
    simbolo = "✓" if diferencia == 0 else "⚠"
    print(f"{simbolo} {estrato_code:<6} {nombre:<20} {esperado:>10} {obtenido:>10} {diferencia:>+12}")

# Distribución urbano/rural
print("\nDistribución Urbano/Rural:")
urbano_pct = (df['ur'] == 1).sum() / len(df) * 100
rural_pct = (df['ur'] == 2).sum() / len(df) * 100
print(f"  Urbano: {urbano_pct:.1f}% (esperado: ~68.3%)")
print(f"  Rural:  {rural_pct:.1f}% (esperado: ~31.7%)")

# ============================================
# 2. DETECCIÓN DE CÓDIGOS ESPECIALES LAPOP
# ============================================
print("\n" + "=" * 70)
print("2. DETECCIÓN DE CÓDIGOS ESPECIALES (missings)")
print("=" * 70)

# Códigos especiales de LAPOP que deben tratarse como missing
codigos_missing = {
    888888: "No sabe",
    988888: "No responde",
    999999: "Inaplicable"
}

variables_clave = ['q2', 'edre', 'q10inc', 'etid', 'boletidnew', 'ocupoit']

print(f"\n{'Variable':<15} {'888888':>10} {'988888':>10} {'999999':>10} {'Total missing':>15}")
print("-" * 70)

for var in variables_clave:
    if var in df.columns:
        counts = {code: (df[var] == code).sum() for code in codigos_missing.keys()}
        total_missing = sum(counts.values()) + df[var].isna().sum()
        print(f"{var:<15} {counts[888888]:>10} {counts[988888]:>10} {counts[999999]:>10} {total_missing:>15}")

# ============================================
# 3. VALIDACIÓN DE RANGOS
# ============================================
print("\n" + "=" * 70)
print("3. VALIDACIÓN DE RANGOS DE VARIABLES")
print("=" * 70)

# Edad debe ser >= 18
if 'q2' in df.columns:
    edad_valida = df[(df['q2'] >= 18) & (df['q2'] < 888888)]
    edad_invalida = df[(df['q2'] < 18) & (df['q2'] < 888888)]
    print(f"\nEdad (Q2):")
    print(f"  ✓ Registros válidos (≥18): {len(edad_valida)}")
    print(f"  {'✓' if len(edad_invalida) == 0 else '✗'} Registros inválidos (<18): {len(edad_invalida)}")
    if len(edad_valida) > 0:
        print(f"  Rango: {edad_valida['q2'].min():.0f} - {edad_valida['q2'].max():.0f} años")

# Nivel educativo (EDRE): 0-6
if 'edre' in df.columns:
    edre_valida = df[(df['edre'] >= 0) & (df['edre'] <= 6)]
    edre_invalida = df[(df['edre'] < 0) | ((df['edre'] > 6) & (df['edre'] < 888888))]
    print(f"\nNivel educativo (EDRE):")
    print(f"  ✓ Registros válidos (0-6): {len(edre_valida)}")
    print(f"  {'✓' if len(edre_invalida) == 0 else '✗'} Registros fuera de rango: {len(edre_invalida)}")

# ============================================
# 4. RESUMEN DE INTEGRIDAD
# ============================================
print("\n" + "=" * 70)
print("4. RESUMEN DE INTEGRIDAD DEL DATASET")
print("=" * 70)

variables_alta_prioridad = {
    'q2': ('Edad', 18, 120),
    'edre': ('Nivel educativo', 0, 6),
    'q10inc': ('Ingreso familiar', 1001, 1015),
    'etid': ('Identidad étnica', 1, 7),
    'ur': ('Urbano/Rural', 1, 2),
}

print(f"\n{'Variable':<15} {'Descripción':<25} {'Válidos':>10} {'Missing':>10} {'Completitud':>12}")
print("-" * 80)

for codigo, (nombre, min_val, max_val) in variables_alta_prioridad.items():
    if codigo in df.columns:
        # Contar válidos (excluyendo códigos especiales y fuera de rango)
        validos = df[
            (df[codigo] >= min_val) & 
            (df[codigo] <= max_val) & 
            (df[codigo] < 888888)
        ]
        n_validos = len(validos)
        n_missing = len(df) - n_validos
        completitud = (n_validos / len(df)) * 100
        
        simbolo = "✓" if completitud >= 80 else "⚠"
        print(f"{simbolo} {codigo:<15} {nombre:<25} {n_validos:>10} {n_missing:>10} {completitud:>11.1f}%")

print("\n" + "=" * 70)
print("VALIDACIÓN COMPLETADA")
print("=" * 70)

VALIDACIÓN DEL DATASET LAPOP BOLIVIA 2023

✓ Dataset cargado: 1706 registros × 208 variables

1. VALIDACIÓN DE DISTRIBUCIÓN MUESTRAL

Distribución por regiones (comparar con reporte técnico):

Estrato Región                 Esperado   Obtenido   Diferencia
----------------------------------------------------------------------
✓ 1001   La Paz                      371        371           +0
✓ 1002   Santa Cruz                  365        365           +0
✓ 1003   Cochabamba                  294        294           +0
✓ 1010   Beni-Pando                  192        192           +0
✓ 1011   Potosí-Oruro                241        241           +0
✓ 1012   Chuquisaca-Tarija           243        243           +0

Distribución Urbano/Rural:
  Urbano: 68.5% (esperado: ~68.3%)
  Rural:  31.5% (esperado: ~31.7%)

2. DETECCIÓN DE CÓDIGOS ESPECIALES (missings)

Variable            888888     988888     999999   Total missing
----------------------------------------------------------------------


PRUEBA DE CARACTERISTICAS BUSCADAS

In [6]:
# ============================================
# 5. VALIDACIÓN DE CARACTERÍSTICAS DEFINIDAS
# ============================================
print("\n" + "=" * 70)
print("5. VALIDACIÓN DE CARACTERÍSTICAS DEMOGRÁFICAS DEFINIDAS")
print("=" * 70)

# Definición de todas las características según marco teórico
caracteristicas_definidas = {
    'Edad': {
        'justificacion': 'Generación del votante',
        'codigos': ['q2'],
        'relevancia': 'Moderada'
    },
    'Nivel educativo': {
        'justificacion': 'Determinante sobre tendencias',
        'codigos': ['edre'],
        'relevancia': 'Alta'
    },
    'Ingreso familiar': {
        'justificacion': 'Condición socioeconómica',
        'codigos': ['q10inc'],
        'relevancia': 'Alta'
    },
    'Identidad étnica': {
        'justificacion': 'Factor de peso en el país',
        'codigos': ['etid'],
        'relevancia': 'Contextual'
    },
    'Pertenencia indígena': {
        'justificacion': 'Identidad política diferenciada',
        'codigos': ['boletidnew', 'boletidnewb'],
        'relevancia': 'Contextual'
    },
    'Ubicación': {
        'justificacion': 'Patrones regionales para el voto',
        'codigos': ['prov', 'ur'],
        'relevancia': 'Alta'
    },
    'Ocupación': {
        'justificacion': 'Clase social y sector económico',
        'codigos': ['ocupoit'],
        'relevancia': 'Alta'
    },
    'Género': {
        'justificacion': 'Brecha de género política',
        'codigos': ['q1tc_r'],
        'relevancia': 'Moderada'
    },
    'Religión': {
        'justificacion': 'Valores y posiciones morales',
        'codigos': ['q3cn', 'q5b'],
        'relevancia': 'Alta'
    },
    'Bienes del hogar': {
        'justificacion': 'Nivel socioeconómico objetivo',
        'codigos': ['r3', 'r4a', 'r6', 'r7', 'r12', 'r15', 'r16', 'r18', 'r18n', 'r27'],
        'relevancia': 'Media'
    },
    'Lengua materna': {
        'justificacion': 'Identidad cultural',
        'codigos': ['leng1'],
        'relevancia': 'Media'
    },
    'Formalidad laboral': {
        'justificacion': 'Inserción económica formal',
        'codigos': ['formal'],
        'relevancia': 'Media'
    },
    'Programas sociales': {
        'justificacion': 'Dependencia del Estado',
        'codigos': ['bolcct1a', 'bolcct1b', 'bolcct1c'],
        'relevancia': 'Contextual'
    },
    'Tamaño de municipio': {
        'justificacion': 'Contexto urbano/rural',
        'codigos': ['estratosec'],
        'relevancia': 'Media'
    },
    'Composición del hogar': {
        'justificacion': 'Estructura familiar',
        'codigos': ['q12cn', 'q12bn'],
        'relevancia': 'Moderada'
    },
    'Consumo de medios': {
        'justificacion': 'Acceso a información',
        'codigos': ['gi0n', 'smedia3n'],
        'relevancia': 'Emergente'
    },
    'Participación social': {
        'justificacion': 'Capital social',
        'codigos': ['cp6', 'cp7', 'cp8', 'cp13'],
        'relevancia': 'Media'
    },
    'Cambio en ingreso': {
        'justificacion': 'Movilidad económica percibida',
        'codigos': ['q10e'],
        'relevancia': 'Emergente'
    },
    'Estado civil': {
        'justificacion': 'Estructura familiar',
        'codigos': ['q11n'],
        'relevancia': 'Moderada'
    },
    'Idioma de padres': {
        'justificacion': 'Transmisión cultural',
        'codigos': ['leng4'],
        'relevancia': 'Media'
    }
}

# Verificar disponibilidad de variables
print("\nDisponibilidad de características en el dataset:")
print(f"\n{'Característica':<25} {'Relevancia':<12} {'Variables':>12} {'Disponibles':>12} {'Estado':<10}")
print("-" * 80)

resumen_disponibilidad = {
    'total_caracteristicas': 0,
    'caracteristicas_completas': 0,
    'caracteristicas_parciales': 0,
    'caracteristicas_ausentes': 0,
    'total_variables': 0,
    'variables_disponibles': 0
}

for caracteristica, info in caracteristicas_definidas.items():
    codigos = [c.lower() for c in info['codigos']]
    disponibles = [c for c in codigos if c in df.columns]
    
    n_total = len(codigos)
    n_disponibles = len(disponibles)
    
    resumen_disponibilidad['total_caracteristicas'] += 1
    resumen_disponibilidad['total_variables'] += n_total
    resumen_disponibilidad['variables_disponibles'] += n_disponibles
    
    # Determinar estado
    if n_disponibles == n_total:
        estado = "✓ Completa"
        resumen_disponibilidad['caracteristicas_completas'] += 1
    elif n_disponibles > 0:
        estado = "⚠ Parcial"
        resumen_disponibilidad['caracteristicas_parciales'] += 1
    else:
        estado = "✗ Ausente"
        resumen_disponibilidad['caracteristicas_ausentes'] += 1
    
    print(f"{caracteristica:<25} {info['relevancia']:<12} {n_total:>12} {n_disponibles:>12} {estado:<10}")

# Resumen ejecutivo
print("\n" + "=" * 80)
print("RESUMEN DE DISPONIBILIDAD")
print("=" * 80)
print(f"\nCaracterísticas totales definidas: {resumen_disponibilidad['total_caracteristicas']}")
print(f"  ✓ Completas (100%):   {resumen_disponibilidad['caracteristicas_completas']} "
      f"({resumen_disponibilidad['caracteristicas_completas']/resumen_disponibilidad['total_caracteristicas']*100:.1f}%)")
print(f"  ⚠ Parciales (>0%):    {resumen_disponibilidad['caracteristicas_parciales']} "
      f"({resumen_disponibilidad['caracteristicas_parciales']/resumen_disponibilidad['total_caracteristicas']*100:.1f}%)")
print(f"  ✗ Ausentes (0%):      {resumen_disponibilidad['caracteristicas_ausentes']} "
      f"({resumen_disponibilidad['caracteristicas_ausentes']/resumen_disponibilidad['total_caracteristicas']*100:.1f}%)")

print(f"\nVariables totales esperadas: {resumen_disponibilidad['total_variables']}")
print(f"Variables disponibles en dataset: {resumen_disponibilidad['variables_disponibles']} "
      f"({resumen_disponibilidad['variables_disponibles']/resumen_disponibilidad['total_variables']*100:.1f}%)")


5. VALIDACIÓN DE CARACTERÍSTICAS DEMOGRÁFICAS DEFINIDAS

Disponibilidad de características en el dataset:

Característica            Relevancia      Variables  Disponibles Estado    
--------------------------------------------------------------------------------
Edad                      Moderada                1            1 ✓ Completa
Nivel educativo           Alta                    1            1 ✓ Completa
Ingreso familiar          Alta                    1            1 ✓ Completa
Identidad étnica          Contextual              1            1 ✓ Completa
Pertenencia indígena      Contextual              2            2 ✓ Completa
Ubicación                 Alta                    2            2 ✓ Completa
Ocupación                 Alta                    1            1 ✓ Completa
Género                    Moderada                1            1 ✓ Completa
Religión                  Alta                    2            2 ✓ Completa
Bienes del hogar          Media                  10